# Source-lesson compatibility check

This notebook exercises the extra PyTorch APIs used by the original Zero-to-Hero lesson and exercise notebooks. Everything below runs inside the Pyodide browser worker.

In [ ]:
%pip install -q torchlite
import torch
import torch.nn as nn
import torch.nn.functional as F

print('torchlite', torch.__version__)

## Makemore lesson operations

`add`, `unbind`, diagnostics, and initialization appear in the original bigram-through-WaveNet notebooks.

In [ ]:
broadcast = torch.add(torch.ones(4, 1), torch.ones(4))
embedding = torch.arange(24).view(2, 3, 4).float().requires_grad_()
joined = torch.cat(torch.unbind(embedding, 1), 1)
joined.sum().backward()

weight = torch.empty(6, 4)
nn.init.xavier_uniform_(weight, generator=torch.Generator().manual_seed(7))

assert broadcast.shape == (4, 4)
assert joined.shape == (2, 12)
assert torch.allclose(embedding.grad, torch.ones_like(embedding))
assert abs(nn.init.calculate_gain('tanh') - 5/3) < 1e-7
print('makemore lesson APIs: ok')

## GPT lesson operations

Browser execution is CPU-only, so CUDA/MPS probes correctly return `False`; sampling, checkpoint I/O, modules, and optimizers still work.

In [ ]:
scores = torch.tensor([[0.1, 0.7, -0.2, 0.4]])
values, indices = torch.topk(scores, 2)
assert indices.tolist() == [[1, 3]]
assert torch.gather(scores, 1, indices).tolist() == values.tolist()
assert not torch.cuda.is_available()
assert not torch.backends.mps.is_available()

model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
loss = F.cross_entropy(model(torch.randn(3, 4)), torch.tensor([0, 1, 0]))
loss.backward()
optimizer.step()

torch.save(model.state_dict(), 'lesson-state.pkl')
restored = torch.load('lesson-state.pkl')
assert set(restored) == set(model.state_dict())
print('gpt lesson APIs: ok')

In [ ]:
from pathlib import Path
Path('lesson-state.pkl').unlink()
print('source-lesson compatibility: ok')